In [2]:
import pandas as pd

df_model = pd.read_csv("data/insurance_with_predictions.csv")

# final pricing choice
base_margin = 0.10
buffer = 2000

df_model['premium_final'] = df_model['predicted_cost'] * (1 + base_margin) + buffer
df_model['profit_final'] = df_model['premium_final'] - df_model['charges']

# pricing error (over / under pricing)
df_model['pricing_error'] = df_model['premium_final'] - df_model['charges']

# residual (prediction error)
df_model['residual'] = df_model['charges'] - df_model['predicted_cost']

# sanity check
print("Average premium:", df_model['premium_final'].mean())
print("Average profit:", df_model['profit_final'].mean())
print("Loss ratio:", (df_model['profit_final'] < 0).mean())


Average premium: 16658.93456545549
Average profit: 3388.5123003142357
Loss ratio: 0.08520179372197309


## Final pricing rule ##

In this project, the final insurance pricing strategy is defined as:
$\text{Premium}_i = \hat{C}_i \times (1 + m) + k$

where:
- $\hat{C}_i$: predicted medical cost for individual $i$
- $m$: margin (profit loading)
- $k$: safety buffer (fixd adjustment for uncertainty)

### Components of the pricing rule ###
1. Predicted cost ($\hat{C}$)

The base of the pricing model is the predicted medical cost, estimated using a regression model.
- Captures expected healthcare expenses
- Reflects individual risk factors (e.g., age, BMI, smoking status)
- Serves as the primary driver of the premium

2. Margin ($m$)

A proportional markup is applied to ensure profitability.
$\hat{C} \times (1 + m)$
- Accounts for administrative costs and profit
- Scales with the level of predicted risk
- Maintains proportional pricing across individuals

3. Safety Buffer ($k$)

A fixed amount is added to all premiums:
$+k$
- Designed to account for model uncertainty
- Helps mitigate underestimation of high-cost cases
- Provides protection against unexpected medical expenses

### Rationale for the final design ###
Earlier analysis showed that:
- Increasing margins significantly improves profit
- However, it has limited impact on reducing loss cases
- Losses are primarily caused by prediction errors, especially in unexpected high-cist individuals

Therefore, a safety buffer is introduced to address this limitation.

### Interpretation ###
The final pricing rule can be interpreted as:
    Premiums are primarily determined by predicted medical costs, with an additional proportional margin for profitability and a fixed safety buffer to account for model uncertainty.

### Key Insight ###
- Margin controls profitability
- Buffer controls risk from prediction error

Together, they balance:
- Financial performance
- Stability to uncertainty

### Limitations ###
- The safet buffer applies uniformly to all individuals
- This may introduce cross-subsidisation, where low-risk individuals contribute more than their expected cost
- Further analysis is required to evaluate fairness across groups.

In [3]:
sample_cases = df_model[['age', 'bmi', 'smoker', 'predicted_cost', 'premium_final', 'charges', 'profit_final']].sample(5, random_state=42)
print(sample_cases)

      age     bmi smoker  predicted_cost  premium_final      charges  \
764    45  25.175     no     9855.093520   12840.602872   9095.06825   
887    36  30.020     no     7804.873583   10585.360941   5272.17580   
890    64  26.885    yes    28972.725662   33869.998228  29330.98315   
1293   46  25.745     no    10108.859112   13119.745024   9301.89355   
259    19  31.920    yes    34778.257534   40256.083287  33750.29180   

      profit_final  
764    3745.534622  
887    5313.185141  
890    4539.015078  
1293   3817.851474  
259    6505.791487  


## Individual explainability ##

In [4]:
# normal case
normal_cases = df_model[['age', 'bmi', 'smoker', 'predicted_cost', 'premium_final', 'charges', 'profit_final']].sample(5, random_state=42)
print("1. Normal cases: ", normal_cases, "\n")

# loss case
loss_cases = df_model[df_model['profit_final'] < 0][['age', 'bmi', 'smoker', 'predicted_cost', 'premium_final', 'charges', 'profit_final']].head(5)
print("2. Loss cases: ", loss_cases, "\n")

# high-profit case
high_profit_cases = df_model[df_model['profit_final'] > 5000][['age', 'bmi', 'smoker', 'predicted_cost', 'premium_final', 'charges', 'profit_final']].head(5)
print("3. High profit cases: ", high_profit_cases, "\n")

1. Normal cases:        age     bmi smoker  predicted_cost  premium_final      charges  \
764    45  25.175     no     9855.093520   12840.602872   9095.06825   
887    36  30.020     no     7804.873583   10585.360941   5272.17580   
890    64  26.885    yes    28972.725662   33869.998228  29330.98315   
1293   46  25.745     no    10108.859112   13119.745024   9301.89355   
259    19  31.920    yes    34778.257534   40256.083287  33750.29180   

      profit_final  
764    3745.534622  
887    5313.185141  
890    4539.015078  
1293   3817.851474  
259    6505.791487   

2. Loss cases:      age     bmi smoker  predicted_cost  premium_final      charges  \
3    33  22.705     no     6726.559686    9399.215655  21984.47061   
9    60  25.840     no    13811.963890   17193.160279  28923.13692   
34   28  36.400    yes    39177.439684   45095.183653  51194.55914   
45   55  37.300     no    12694.127130   15963.539843  20630.28351   
62   64  24.700     no    14892.254133   18381.479547  

## Sample Cases ##
### normal 케이스 ###
### loss 케이스 ###
### high-profit case ###